# 11주차 팀 과제 (9팀): DAG와 복합표본 회귀
국민건강영양조사 제9기(2022-2024) 중 2024년 자료. 아침 결식(이분형)과 비만 유병의 관련성을 무보정(모형 1)과 DAG 보정(모형 2)으로 비교한다.
- 1~2절은 10주차 노트북(`10_table1.ipynb`)의 불러오기·변수 만들기·`in_study` 코드를 그대로 사용한다.
- 3절부터가 11주차 내용이다. 모든 숫자는 이 노트북의 출력에서 가져온다.


## 1. 내 Google Drive 연결 및 자료 읽기 (10주차와 같은 코드)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/hdac_data/HN24_ALL(SAS)'
DATA_FILE = f'{DATA_DIR}/hn24_all.sas7bdat'

In [ ]:
!pip -q install pyreadstat
import pyreadstat, numpy as np, pandas as pd

need = ['age', 'sex', 'HE_prg', 'L_BR_FQ', 'HE_ht', 'HE_wt', 'HE_BMI', 'HE_obe',
        'wt_tot', 'kstrata', 'psu', 'town_t', 'educ', 'sm_presnt', 'dr_month', 'N_DIET', 'N_EN']

_, meta0 = pyreadstat.read_sas7bdat(DATA_FILE, metadataonly=True)
lower_map = {c.lower(): c for c in meta0.column_names}
present = {v: lower_map.get(v.lower()) for v in need}
assert all(present.values()), [v for v, a in present.items() if a is None]

df, meta = pyreadstat.read_sas7bdat(DATA_FILE, usecols=list(present.values()))
df = df.rename(columns={a: v for v, a in present.items()})
print(df.shape)

## 2. 노출·결과·공변량 만들기와 분석 대상 표시 (10주차와 같은 코드)
원본 행은 지우지 않고 표시변수만 만든다.

- 노출: `L_BR_FQ` 3·4 = 결식(1), 1·2 = 아침 식사(0), 9·결측 = 결측 (이용지침서 218쪽)
- 결과: `HE_obe` 4~6 = 비만(BMI 25 이상) (이용지침서 179쪽)
- 공변량: `educ` 88·99, `N_DIET` 9는 결측 처리 (이용지침서 134·232쪽)
- 교육수준 묶음(`edu4`)은 팀이 확정한 기준으로 바꿀 수 있다.

In [ ]:
df['breakfast_skip'] = df['L_BR_FQ'].map({1: 0, 2: 0, 3: 1, 4: 1})
df['obese'] = np.where(df['HE_obe'].isna(), np.nan, (df['HE_obe'] >= 4).astype(float))
df['educ_c'] = df['educ'].where(~df['educ'].isin([88, 99]))
df['edu4'] = df['educ_c'].map({1: 1, 2: 1, 3: 1, 4: 2, 5: 3, 6: 4, 7: 4, 8: 4})   # 묶음은 팀이 확정
df['N_DIET_c'] = df['N_DIET'].where(~df['N_DIET'].isin([9]))
df['clu'] = df['kstrata'].astype(str) + '_' + df['psu'].astype(str)   # 집락 식별자 (층 + 조사구)

In [ ]:
conds = [
    ('1. 만 19세 이상', df['age'] >= 19, '연구 대상 아님(비해당)'),
    ('2. 임신 중 아님', df['HE_prg'] != 1, '제외 기준(임신)'),
    ('3. 아침식사 빈도 응답', df['L_BR_FQ'].isin([1, 2, 3, 4]), '결측(노출 정의 불가)'),
    ('4. 신장·체중·BMI 값 있음', df[['HE_ht', 'HE_wt', 'HE_BMI']].notna().all(axis=1),
     '결측(BMI 이상값 기준은 적용하지 않음)'),
    ('5. 가중치·층·집락 유효',
     df['wt_tot'].notna() & (df['wt_tot'] > 0) & df['kstrata'].notna() & df['psu'].notna(),
     '설계상 제외'),
]

cur = pd.Series(True, index=df.index)
rows = [{'단계': '0. 전체 조사 참여자', '제외 인원': 0, '남은 인원': int(cur.sum()), '사유': '출발 대상'}]
for name, cond, why in conds:
    before = int(cur.sum())
    cur = cur & cond
    rows.append({'단계': name, '제외 인원': before - int(cur.sum()), '남은 인원': int(cur.sum()), '사유': why})

df['in_study'] = cur.astype(int)      # 분석 대상 표시변수
df['design_ok'] = conds[-1][1]        # 설계 변수 유효 표시변수
flow = pd.DataFrame(rows)
print(flow.to_string(index=False))

base = (df['age'] >= 19) & (df['HE_prg'] != 1) & df['L_BR_FQ'].isin([1, 2, 3, 4])
print()
print('4단계 변수별 결측:', df.loc[base, ['HE_ht', 'HE_wt', 'HE_BMI']].isna().sum().to_dict())

In [ ]:
d = df[df['design_ok']].copy()        # 설계 정보가 있는 전체 (도메인 분석용)
print('설계 자료 n:', len(d), '/ 분석 대상(도메인) n:', int(d['in_study'].sum()))
print('층 수:', d['kstrata'].nunique(), '/ 집락(PSU) 수:', d['clu'].nunique())
psu_per_strata = d.groupby('kstrata')['clu'].nunique()
print('층당 PSU 수 최소:', int(psu_per_strata.min()), '(2 이상이어야 분산 추정 가능)')
print('가중치 합(추정 모집단, 명):', round(d['wt_tot'].sum()))

## 3. 보정변수가 모두 있는 사람(완전 사례) 표시
모형 1과 모형 2는 **같은 사람**으로 적합해야 계수의 변화를 비교할 수 있다. 행은 지우지 않고 표시변수 `cc`만 만든다.

보정변수 목록은 DAG(C-1)에서 정한 최소 보정집합과 같아야 한다. 아래 목록은 10주차 노트북에서 읽은 변수 중 DAG의 교란변수에 해당하는 것이다.
**소득·신체활동**은 팀 DAG의 교란 후보이지만 10주차 노트북의 읽기 목록(`need`)에 없다. 정의표에 있고 원자료에도 있으면 1절 `need`에 추가한 뒤 아래 목록에 넣고, 없으면 보정하지 못한 교란으로 C-3 한계에 적는다.


In [ ]:
# 보정변수 (DAG의 최소 보정집합과 일치시킬 것)
ADJ_CONT = ['age']                                   # 연속형: 1세 단위
ADJ_CAT  = ['sex', 'town_t', 'edu4', 'sm_presnt', 'dr_month']   # 범주형
# ADJ_CAT += ['소득변수', '신체활동변수']              # ← 정의표·원자료에 있을 때만 추가
ADJ_ALL = ADJ_CONT + ADJ_CAT

need_cols = ['breakfast_skip', 'obese'] + ADJ_ALL
d['in_study'] = d['in_study'].astype(int)
d['cc'] = ((d['in_study'] == 1) & d[need_cols].notna().all(axis=1)).astype(int)   # 완전 사례 표시

n_t1 = int(d['in_study'].sum())
n_cc = int(d['cc'].sum())
print(f'Table 1 분석 대상: {n_t1:,}명')
print(f'완전 사례(보정변수 모두 있음): {n_cc:,}명 (차이 {n_t1 - n_cc:,}명)')
print()
print('분석 대상 안에서 변수별 결측 인원:')
print(d.loc[d['in_study'] == 1, need_cols].isna().sum().to_string())
print()
print('완전 사례의 노출 분포(비가중 n):', d.loc[d['cc'] == 1, 'breakfast_skip'].value_counts().sort_index().to_dict(), ' (0=비결식군, 1=결식군)')
print('완전 사례의 비만 비율(비가중 %):', round(100 * d.loc[d['cc'] == 1, 'obese'].mean(), 1))


## 4. 설계 기반 로지스틱 회귀 함수
가중치(`wt_tot`)로 계수를 추정하고, 층(`kstrata`)·집락(`clu` = 층 + 조사구)을 반영한 Taylor 선형화로 표준오차를 구한다(10주차 Table 1과 같은 방식).
도메인은 행을 지우지 않고 **분석에 쓰지 않는 사람의 가중치를 0**으로 두어 처리한다. 95% 신뢰구간과 p값은 설계 자유도(집락 수 − 층 수)의 t분포를 쓴다.

> 6주차 실습의 회귀 함수가 있으면 같은 입력으로 한 번 돌려 이 함수의 결과와 대조한다. 소수점 아래 값이 조금 다르면 표준오차의 소표본 보정 방식 차이일 수 있으므로, 최종 표에는 팀이 쓰기로 한 함수의 출력을 일관되게 사용한다.


In [ ]:
from scipy import stats


def make_X(df, exposure, cont=(), cat=(), cat_ref=None, rows=None):
    """설계행렬 만들기. 상수항 + 노출 + 연속형 + 범주형(더미, 기준범주는 cat_ref로 지정).
    결측이 있는 행은 NaN으로 남겨 두므로, 사용할 행은 호출하는 쪽에서 고른다.
    rows: 범주 수준을 찾을 행(bool). 분석에 쓰는 행만 주면, 그 행에 없는 수준의 빈 더미가 생기지 않는다."""
    cat_ref = cat_ref or {}
    X = pd.DataFrame(index=df.index)
    X['const'] = 1.0
    X[exposure] = df[exposure].astype(float)
    for c in cont:
        X[c] = df[c].astype(float)
    for c in cat:
        levels = sorted((df[c] if rows is None else df.loc[rows, c]).dropna().unique())
        ref = cat_ref.get(c, levels[0])
        for lev in levels:
            if lev == ref:
                continue
            X[f'{c}={lev:g}'] = np.where(df[c].isna(), np.nan, (df[c] == lev).astype(float))
    return X


def design_df(df, strata, psu):
    """설계 자유도 = 집락 수 - 층 수 (전체 설계 기준)."""
    n_psu = df[[strata, psu]].drop_duplicates().shape[0]
    n_str = df[strata].nunique()
    return n_psu - n_str


def svy_logit(df, y, X, w, strata, psu, use, maxit=50, tol=1e-9):
    """설계 기반 가중 로지스틱 회귀 (Taylor 선형화 분산).
    df 전체 행을 유지하고, use(bool)가 False인 행은 가중치 0으로 둔다(도메인 분석).
    - 점수: u_i = w_i * x_i * (y_i - p_i)
    - 분산: A^-1 B A^-1, B는 층 안에서 집락 합의 편차로 n_h/(n_h-1) * sum (t_hj - tbar_h)(t_hj - tbar_h)'
    """
    use = np.asarray(use, bool)
    names = list(X.columns)
    Xv = X.to_numpy(float).copy()
    yv = df[y].to_numpy(float).copy()
    wv = df[w].to_numpy(float).copy()

    bad = use & (np.isnan(Xv).any(axis=1) | np.isnan(yv) | np.isnan(wv))
    if bad.any():
        raise ValueError(f'use=True인데 결측이 있는 행 {int(bad.sum())}개: use를 완전 사례로 만드세요.')

    wd = np.where(use, wv, 0.0)
    wd = wd / wd[use].mean()                 # 척도만 바꿈(추정값과 SE에는 영향 없음)
    Xv[~use] = 0.0
    yv[~use] = 0.0

    k = Xv.shape[1]
    beta = np.zeros(k)
    converged = False
    for _ in range(maxit):
        eta = Xv @ beta
        p = 1.0 / (1.0 + np.exp(-eta))
        grad = Xv.T @ (wd * (yv - p))
        A = (Xv * (wd * p * (1 - p))[:, None]).T @ Xv
        step = np.linalg.solve(A, grad)
        beta = beta + step
        if np.max(np.abs(step)) < tol:
            converged = True
            break
    eta = Xv @ beta
    p = 1.0 / (1.0 + np.exp(-eta))
    A = (Xv * (wd * p * (1 - p))[:, None]).T @ Xv
    U = Xv * (wd * (yv - p))[:, None]          # 행별 점수 (도메인 밖은 0)

    key = pd.DataFrame(U, columns=names)
    key['h'] = df[strata].to_numpy()
    key['c'] = df[psu].to_numpy()
    tot = key.groupby(['h', 'c'])[names].sum().reset_index()
    B = np.zeros((k, k))
    lonely = 0
    for _, g in tot.groupby('h'):
        n_h = len(g)
        if n_h < 2:
            lonely += 1
            continue
        D = g[names].to_numpy() - g[names].to_numpy().mean(axis=0)
        B += n_h / (n_h - 1) * D.T @ D
    Ainv = np.linalg.inv(A)
    V = Ainv @ B @ Ainv
    se = np.sqrt(np.diag(V))
    dfree = design_df(df, strata, psu)
    return {'names': names, 'beta': beta, 'se': se, 'V': V, 'df': dfree,
            'n': int(use.sum()), 'converged': converged, 'lonely': lonely,
            'max_abs_beta': float(np.max(np.abs(beta)))}


def or_table(res, terms=None, level=0.95):
    """계수를 오즈비 (95% CI), p값으로 변환. 신뢰구간과 p값은 t분포(설계 자유도) 사용."""
    tcrit = stats.t.ppf(1 - (1 - level) / 2, res['df'])
    rows = []
    for i, nm in enumerate(res['names']):
        if nm == 'const' or (terms is not None and nm not in terms):
            continue
        b, s = res['beta'][i], res['se'][i]
        p = 2 * (1 - stats.t.cdf(abs(b / s), res['df']))
        rows.append({'항': nm, 'beta': b, 'SE': s, 'OR': np.exp(b),
                     'CI하': np.exp(b - tcrit * s), 'CI상': np.exp(b + tcrit * s), 'p': p})
    return pd.DataFrame(rows)


## 5. 모형 1·모형 2 적합
- 노출: `breakfast_skip` (1 = 결식군, 0 = 비결식군[기준])
- 결과: `obese` (1 = 비만, BMI 25 이상)
- 모형 1: 노출만 / 모형 2: 노출 + 보정집합. **둘 다 완전 사례(`cc == 1`)** 로 적합한다.
- 범주형 보정변수의 기준 범주는 `cat_ref`에 적는다(아래는 가장 작은 코드값을 기준으로 둔 예시이므로 DAG 문서의 설명과 맞게 고친다).


In [ ]:
use = d['cc'].to_numpy() == 1
cat_ref = {'sex': 1, 'town_t': 1, 'edu4': 1, 'sm_presnt': 0, 'dr_month': 0}

X1 = make_X(d, 'breakfast_skip')
X2 = make_X(d, 'breakfast_skip', cont=ADJ_CONT, cat=ADJ_CAT, cat_ref=cat_ref, rows=use)

m1 = svy_logit(d, 'obese', X1, 'wt_tot', 'kstrata', 'clu', use)
m2 = svy_logit(d, 'obese', X2, 'wt_tot', 'kstrata', 'clu', use)

for name, m in [('모형 1', m1), ('모형 2', m2)]:
    print(f"{name}: 분석 인원 {m['n']:,}명, 수렴 {m['converged']}, 설계 자유도 {m['df']}, 최대|계수| {m['max_abs_beta']:.2f}")
assert m1['n'] == m2['n'], '모형 1과 모형 2의 분석 인원이 다릅니다'
assert m1['converged'] and m2['converged'], '수렴하지 않았습니다'

print()
print('모형 2 설계행렬 열(기준 범주가 맞는지 확인):', m2['names'])


## 6. Table 2 형식으로 정리
노출 행만 표에 넣는다(보정변수의 계수는 넣지 않아도 된다). OR는 비결식군 대비 결식군이며 95% 신뢰구간과 p값은 설계 기반 추정이다.


In [ ]:
def one(m, label):
    t = or_table(m, terms=['breakfast_skip']).iloc[0]
    return {'모형': label, '분석 인원(n)': f"{m['n']:,}",
            'OR (95% CI)': f"{t['OR']:.2f} ({t['CI하']:.2f}–{t['CI상']:.2f})",
            'p': ('<0.001' if t['p'] < 0.001 else f"{t['p']:.3f}")}

table2 = pd.DataFrame([one(m1, '모형 1 (무보정)'), one(m2, '모형 2 (DAG 보정집합)')])
print(table2.to_string(index=False))

print()
print(f"Table 1 분석 대상 {n_t1:,}명 대비 회귀 분석 인원 {m1['n']:,}명 (보정변수 결측으로 {n_t1 - m1['n']:,}명 제외)")
print()
print('참고: 로그 오즈비(계수)와 표준오차')
print(or_table(m1, terms=['breakfast_skip'])[['항', 'beta', 'SE']].round(4).to_string(index=False))
print(or_table(m2, terms=['breakfast_skip'])[['항', 'beta', 'SE']].round(4).to_string(index=False))


In [ ]:
# 문서(docs/11_regression.md)에 붙여 넣을 마크다운 표
print(table2.to_markdown(index=False))


## 7. 해석 전 확인
- 비만 유병이 높은(약 37%) 결과에서는 오즈비가 위험비(유병률비)보다 1에서 더 멀어 보인다. 오즈비를 "위험이 몇 배"로 읽지 않는다. 유병률비는 12주차에 다룬다.
- 한 시점 자료이므로 결과는 "관련성"으로 해석한다.
